# Stage-4 v2 — APTOS cache generation and v2 bundle (research record §46)

Run ONCE per trained Stage-4 model, AFTER `stage4_v2_training.ipynb` (training, export, one-time IDRiD gate). GPU runtime.

1. Setup. 2. Inputs: exported model (SHA-verified), gate report, the 3,651-image population.
3. Stage 3: 25-image LWNet parity (≤ 1e-4) → copy the verified vessel maps into `cache/Stage3/s3-91f0cada/` (+ C2 V-pyramid).
4. RGB: 25-image parity (≤ 1e-6) → copy the canonical 512 RGB into `cache/Stage2/rgb-v1/`.
5. Stage 4: deterministic inference on all 3,651 images → `cache/Stage4/s4v2-<sha12>-K4/` (uint8 512×512×8), freshness canary, full verification (+ C2 Q-pyramid).
6. Bundle manifest + independent verification.

Every step is resumable (rerun the notebook). Legacy Stage-4 lesion caches are never read; a completed generation is never rewritten.
Expected runtime: copies ~1 h (Drive I/O), Stage-4 inference ~1–1.5 h on a T4.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import stage4_v2_setup
SETUP_INFO = stage4_v2_setup.setup_stage4_v2(fetch_weights=False)
import json, posixpath
import numpy as np, torch
import colab_config, config
import multiseed_runs as msr
import pipeline_v2_config as v2cfg
import stage34_cache_v2 as cache
import stage4_v2 as s4
import stage4_v2_aptos_cache as ac
import stage4_v2_data as data
assert torch.cuda.is_available(), "GPU runtime required for Stage-4 inference"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: exported Stage-4 model, gate report, population ====
STAGE4_RUN_DIR = posixpath.join(colab_config.DRIVE.experiments_root, "Stage4V2", "stage4v2_k4_seed42_bs8")
GATE_OVERRIDE_TOKEN = None        # only if the model FAILED the gate and you deliberately proceed (recorded)
EXPORT = json.load(open(posixpath.join(STAGE4_RUN_DIR, "exported_model.json")))
MODEL_PATH, MODEL_SHA = EXPORT["model_path"], EXPORT["model_sha256"]
assert cache.sha256_file(MODEL_PATH) == MODEL_SHA, "exported model does not match its recorded SHA"
GATE = ac.require_gate(posixpath.join(os.path.dirname(MODEL_PATH), "idrid_test_gate.json"), MODEL_SHA, GATE_OVERRIDE_TOKEN)
TRAIN_E, VAL_E, SPLIT_SHA = msr.verify_split()
POP = ac.aptos_population(TRAIN_E, VAL_E, SPLIT_SHA)
IDS = ac.population_ids(POP)
GEN4 = cache.stage4_generation_id(MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
LEGACY_LOOSE = config.LOCAL_FEATURE_RESULTS_DIR               # only loose _vessel_ / _rgb_ files are read from here
APTOS_PROCESSED = os.path.join(config.dataset_processed_dir("APTOS2019"), "train_images")
missing = [i for i in IDS if not os.path.exists(os.path.join(APTOS_PROCESSED, f"{i}.png"))]
assert not missing, f"Stage-2 outputs missing for {len(missing)} images, e.g. {missing[:5]}"


def load_native(i):
    return data.read_rgb(os.path.join(APTOS_PROCESSED, f"{i}.png"))


print("model", MODEL_SHA, "| generation", GEN4, "| gate", GATE["PASS"], "| population", len(IDS), POP["population_sha256"])

In [ ]:
# ==== [3] STAGE 3: parity on 25 fixed ids, then copy + verify (+ C2 V-pyramid) ====
from vessel_segmentation_inference import DEFAULT_MODEL_PATH as LWNET_PATH, load_vessel_model
S3_SHA = ac.assert_lwnet(LWNET_PATH)
S3_DIR = ac.data_dir("stage3_cache_v2", v2cfg.STAGE3_GENERATION)
S3_MANIFEST_PATH = ac.manifest_path("stage3_cache_v2", v2cfg.STAGE3_GENERATION)
if os.path.exists(S3_MANIFEST_PATH):
    S3 = cache.load_manifest(S3_MANIFEST_PATH, expected_kind="stage3_cache_v2", expected_generation_id=v2cfg.STAGE3_GENERATION)
    assert S3["parity"]["passed"] and set(S3["files"]) >= set(IDS)
    print("Stage-3 generation exists; verifying")
else:
    LWNET = load_vessel_model(LWNET_PATH)
    PARITY3 = ac.run_parity(ac.parity_ids(IDS, v2cfg.STAGE3_PARITY_N),
                            lambda i: np.load(cache.assert_legacy_vessel_source(os.path.join(LEGACY_LOOSE, cache.vessel_filename(i)))),
                            lambda i: ac.recompute_vessel_512(load_native(i), LWNET), v2cfg.STAGE3_PARITY_TOL, "stage3")
    print("Stage-3 parity:", PARITY3["max_abs_delta"], "<=", PARITY3["tol"])
    SHA3 = ac.copy_reused_generation("stage3_cache_v2", IDS, LEGACY_LOOSE, S3_DIR)
    S3 = cache.stage3_cache_manifest(lwnet_sha256=S3_SHA, tta=True, parity=PARITY3, split_sha256=SPLIT_SHA, files=SHA3)
VFEAT = ac.verify_reused_generation("stage3_cache_v2", IDS, S3_DIR, S3["files"], features=True)
ac.write_generation_manifest("stage3_cache_v2", v2cfg.STAGE3_GENERATION, S3)
V_PYR = posixpath.join(os.path.dirname(S3_DIR), "c2_v_pyramid.npz")
if not os.path.exists(V_PYR):
    ac.save_pyramid_features(V_PYR, VFEAT, IDS, ("vessel",), {"stage3_sha256": S3_SHA, "generation_id": v2cfg.STAGE3_GENERATION,
                                                             "population_sha256": POP["population_sha256"]})
print("Stage 3 OK:", len(S3["files"]), "files; V-pyramid", V_PYR)

In [ ]:
# ==== [4] RGB: parity on 25 fixed ids, then copy + verify ====
S2_DIR = ac.data_dir("stage2_rgb_v2", v2cfg.STAGE2_RGB_GENERATION)
S2_MANIFEST_PATH = ac.manifest_path("stage2_rgb_v2", v2cfg.STAGE2_RGB_GENERATION)
if os.path.exists(S2_MANIFEST_PATH):
    S2 = cache.load_manifest(S2_MANIFEST_PATH, expected_kind="stage2_rgb_v2", expected_generation_id=v2cfg.STAGE2_RGB_GENERATION)
    assert S2["parity"]["passed"] and set(S2["files"]) >= set(IDS)
else:
    PARITY2 = ac.run_parity(ac.parity_ids(IDS, v2cfg.STAGE3_PARITY_N),
                            lambda i: np.load(cache.assert_legacy_rgb_source(os.path.join(LEGACY_LOOSE, cache.rgb_filename(i)))),
                            lambda i: ac.recompute_rgb_512(load_native(i)), v2cfg.RGB_PARITY_TOL, "rgb")
    print("RGB parity:", PARITY2["max_abs_delta"], "<=", PARITY2["tol"])
    SHA2 = ac.copy_reused_generation("stage2_rgb_v2", IDS, LEGACY_LOOSE, S2_DIR)
    S2 = dict(cache.stage2_cache_manifest(split_sha256=SPLIT_SHA, files=SHA2), parity=PARITY2)
ac.verify_reused_generation("stage2_rgb_v2", IDS, S2_DIR, S2["files"])
ac.write_generation_manifest("stage2_rgb_v2", v2cfg.STAGE2_RGB_GENERATION, S2)
print("RGB OK:", len(S2["files"]), "files")

In [ ]:
# ==== [5] STAGE 4: deterministic inference -> s4v2-<sha12>-K4 (+ canary, verification, C2 Q-pyramid) ====
S4_MANIFEST_PATH = ac.manifest_path("stage4_cache_v2", GEN4)
if os.path.exists(S4_MANIFEST_PATH):
    S4 = cache.load_manifest(S4_MANIFEST_PATH, expected_kind="stage4_cache_v2", expected_generation_id=GEN4)
    assert S4["stage4_sha256"] == MODEL_SHA
    print("Stage-4 generation is complete (immutable); verifying")
    QFEAT = ac.verify_stage4_generation(IDS, MODEL_SHA, S3_SHA, S4["files"])
else:
    DET = ac.configure_determinism()
    MODEL = s4.load_stage4_v2(MODEL_PATH, expected_sha256=MODEL_SHA, classes=v2cfg.STAGE4_V2A_CLASSES).to("cuda")
    compute = lambda rgb: s4.pathology_cache_maps(MODEL, rgb, device="cuda", amp=True)
    SHA4 = ac.generate_stage4_maps(MODEL, MODEL_SHA, IDS, load_native, S3_SHA, device="cuda", amp=True, compute_maps=compute)
    CANARY = ac.freshness_canary(IDS, MODEL_SHA, S3_SHA, load_native, compute)
    QFEAT = ac.verify_stage4_generation(IDS, MODEL_SHA, S3_SHA, SHA4)
    model_manifest = posixpath.join(os.path.dirname(MODEL_PATH), "stage4_model_manifest.json")
    S4 = dict(cache.stage4_cache_manifest(stage4_sha256=MODEL_SHA, stage3_sha256=S3_SHA, classes=v2cfg.STAGE4_V2A_CLASSES,
                                          split_sha256=SPLIT_SHA, files=SHA4,
                                          model_manifest_sha256=cache.sha256_file(model_manifest)),
              canary=CANARY, stage4_gate=GATE, population_sha256=POP["population_sha256"],
              inference={"device": torch.cuda.get_device_name(0), "amp": True, **DET, "torch": torch.__version__,
                         "geometry": "full-frame direct resize 1536 -> exact 3x3 mean+max -> 512 uint8"},
              git_commit=REPO_COMMIT)
    ac.write_generation_manifest("stage4_cache_v2", GEN4, S4)
Q_PYR = posixpath.join(os.path.dirname(ac.data_dir("stage4_cache_v2", GEN4)), "c2_q_pyramid.npz")
if not os.path.exists(Q_PYR):
    ac.save_pyramid_features(Q_PYR, QFEAT, IDS, cache.channel_names(v2cfg.STAGE4_V2A_CLASSES),
                             {"stage4_sha256": MODEL_SHA, "generation_id": GEN4, "stage3_sha256": S3_SHA,
                              "population_sha256": POP["population_sha256"]})
assert len(S4["files"]) == v2cfg.APTOS_POPULATION
print("Stage 4 OK:", len(S4["files"]), "maps; canary", S4["canary"], "; Q-pyramid", Q_PYR)

In [ ]:
# ==== [6] BUNDLE + independent verification ====
import arch1_data as ad
BUNDLE = ac.bundle_manifest_v2(S2, S3, S4, POP)
ac.write_bundle(BUNDLE)
B = ad.Arch1Bundle(expected_bundle_id=BUNDLE["bundle_id"], expected_split_sha256=v2cfg.SPLIT_SHA256,
                   expected_stage4_sha256=MODEL_SHA)
B.require(IDS)
_ = [B.load_sample(i) for i in (B.train_ids[0], B.val_ids[0])]
print(json.dumps(B.describe(), indent=1))
RC = "C:/Users/nsyas/Downloads/rclone-v1.75.0-windows-amd64/rclone-v1.75.0-windows-amd64/rclone.exe"
rel = lambda p: p.split("/MyDrive/", 1)[1]
print("\nNEXT -- C2 on the laptop (Git Bash, repo root):")
print(f'  "{RC}" copy "gdrive:{rel(Q_PYR)}" results/C2_inputs/')
print(f'  "{RC}" copy "gdrive:{rel(V_PYR)}" results/C2_inputs/')
print(f'  "{RC}" copy "gdrive:DiabeticRetinopathy/experiments/{v2cfg.C2_R_FEATURES_REL}" results/C2_inputs/')
print(f"  .venv/Scripts/python stage4_v2_c2.py --r results/C2_inputs/hr_screen_features.npz "
      f"--q results/C2_inputs/c2_q_pyramid.npz --v results/C2_inputs/c2_v_pyramid.npz --stage4-sha {MODEL_SHA}")
print("Then upload the C2 run folder (architecture1_training.ipynb cell 2 reads C2_RESULTS from Drive):")
print(f'  "{RC}" copy results/C2/<c2 run folder> "gdrive:DiabeticRetinopathy/experiments/C2/<c2 run folder>"')